# 02b — Driver vs. Non-Driver Labeling (Unmatched Negative Baseline)

Standalone companion to `02_driver_labeling_pipeline.ipynb`. That notebook's main path
optionally applies **length/expression-matched negative sampling** (`config.USE_MATCHED_NEGATIVE_SAMPLING`)
so non-driver genes aren't just "whatever survived the filters" but are matched to the
positives' covariate distribution.

This notebook deliberately **skips that matching step** and keeps every candidate that
survives the mutation-frequency and disease-pathway filters as-is. The point is to produce
a second, "unmatched" label set as a baseline to compare against — e.g. to check how much
of a downstream model's performance depends on the negative class being covariate-matched
versus just being "easy" (genes that happen to look nothing like drivers on length/expression,
not just on driver status).

**Output is written to a separate file** (`gene_labels_unmatched.csv`, not
`config.GENE_LABELS_FILE`) so running this notebook can never silently overwrite the matched
labels produced by notebook 02.


In [ ]:
import os
import random
import sys
from pathlib import Path

import pandas as pd

# Notebooks live in notebooks/, but config.py's paths (e.g. "data/...") are
# relative to the repo root -- chdir there so those paths resolve correctly
# regardless of where Jupyter's working directory starts out.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
    load_driver_genes,
)
from src.gene_universe import build_protein_coding_gene_universe
from src.negative_sampling import build_driver_exclusion_set
from src.pathway_filter import build_disease_pathway_genes

random.seed(config.RANDOM_SEED)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Where this notebook's (unmatched) labels get saved -- deliberately NOT
# config.GENE_LABELS_FILE, so this can never clobber notebook 02's output.
UNMATCHED_LABELS_FILE = config.PROCESSED_DIR / "gene_labels_unmatched.csv"

print(f"[CONFIG] USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"[CONFIG] USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")
print(f"[CONFIG] Output (this notebook): {UNMATCHED_LABELS_FILE}")


## How the non-driver (negative) set is built

Four stages, each narrowing the candidate pool further:

1. **Start from the full protein-coding gene universe** (GENCODE, `gene_type == "protein_coding"`
   only -- non-coding biotypes like lncRNAs/pseudogenes are excluded from the universe entirely,
   not just from the negative set).
2. **Subtract a broad driver-exclusion set**, not just the 763 CGC positives used as labels.
   `build_driver_exclusion_set()` unions up to four independent driver-gene resources -- NCG,
   CGC, IntOGen, and Bailey et al. 2018 (whichever files are available) -- so a gene doesn't
   have to be in the official positive-label set to be excluded from the negatives; it just has
   to appear on *any* recognized driver list. This is meant to avoid quietly mislabeling
   plausible-but-uncurated driver genes as negatives.
   `non_drivers = all_genes − driver_exclusion_set`.
3. **Remove frequently-mutated genes** (if `config.USE_MUTATION_FILTER`): genes mutated above
   `config.MUTATION_FREQUENCY_THRESHOLD` in *any* TCGA cancer type are dropped -- high mutation
   frequency is itself suggestive of driver-like or at least passenger-enriched behavior, so
   keeping such genes as confident "negatives" would be questionable.
4. **Remove disease-pathway genes** (if `config.USE_PATHWAY_FILTER`): genes belonging to a
   Reactome pathway that's a descendant of the top-level *Disease* pathway are dropped, on
   similar grounds -- pathway involvement in disease processes is itself a driver-like signal.

Whatever protein-coding genes survive all four stages become the **non-driver / negative
class**. Unlike notebook 02's main path, this notebook stops here -- it does **not** apply
length/expression-matched sampling on top, so the negative set here is simply "everything
that passed the filters," not resampled to match the positives' covariate distribution.


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"`.


In [ ]:
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")


## 2. Driver genes and the expanded exclusion set

Positives come from the COSMIC Cancer Gene Census. The exclusion set used to derive raw
non-driver candidates is broader -- NCG + CGC + IntOGen + Bailey et al. 2018 -- so we don't
accidentally keep a gene as a "negative" just because it's missing from CGC specifically.


In [ ]:
driver_genes = load_driver_genes("../../data/Census_allWed.tsv")
print(f"[DRIVERS] {len(driver_genes)} CGC driver genes")

driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

non_drivers = all_genes - driver_exclusion_set
print(
    f"[NON-DRIVERS] {len(non_drivers)} raw candidates "
    "(excluded from NCG/CGC/IntOGen/Bailey)"
)


## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Skipped entirely if `config.USE_MUTATION_FILTER` is `False`.


In [ ]:
if config.USE_MUTATION_FILTER:
    before = len(non_drivers)
    non_drivers = apply_mutation_frequency_filter(
        non_drivers,
        config.MUTATION_FREQUENCY_FILE,
        threshold=config.MUTATION_FREQUENCY_THRESHOLD,
    )
    print(
        f"[MUTATION FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[MUTATION FILTER] skipped (config.USE_MUTATION_FILTER is False)")


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway. Skipped entirely if `config.USE_PATHWAY_FILTER` is `False`.


In [ ]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[PATHWAY FILTER] skipped (config.USE_PATHWAY_FILTER is False)")

print(
    f"[FINAL NEGATIVES] {len(non_drivers)} candidates retained "
    "(unmatched -- no covariate matching applied)"
)


## 5. Build and save the (unmatched) labeled gene table

Saved to a file distinct from notebook 02's output, so this never overwrites the matched
label set.


In [ ]:
labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())

labels_df.to_csv(UNMATCHED_LABELS_FILE, index=False)

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes_unmatched.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")

print(f"[DONE] Labels: {UNMATCHED_LABELS_FILE}")
print(f"[DONE] Drivers: {driver_path}")
print(f"[DONE] Non-drivers: {nondriver_path}")

labels_df.head()
